#### Preparing Data

In [27]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [28]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/ALLMAT_200k_training_data.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [29]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

Number of rows before cleaning: 200000


In [30]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()

# Applying log transformations
df['capacity'] = np.log(df['capacity'])
df['ads_time'] = np.log(df['ads_time'])
df['C_o'] = np.log(df['C_o'])
df['dp'] = np.log(df['dp'])
df['qm1'] = np.log(df['qm1'])
df['kl1'] = np.log(df['kl1'])
df['sorbent_cost_per_kg'] = np.log(df['sorbent_cost_per_kg'])
df['total_LCP'] = np.log(df['total_LCP'])
df['emissions'] = np.log(df['emissions'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

capacity                 2.304194
diameter                 0.955110
column_height            1.000010
porosity                 0.350001
ads_time                 3.401257
C_o                     -6.815012
rho_s                  500.004000
dp                      -8.111725
qm1                     -6.775754
kl1                      0.003334
sorbent_cost_per_kg      0.003125
total_LCP                9.997128
emissions                5.171745
PFAS_eff               -46.051702
PFAS_ads                 0.000022
dtype: float64
capacity                  8.517189
diameter                  4.000000
column_height             3.999990
porosity                  0.499998
ads_time                  7.509330
C_o                       2.302583
rho_s                  1999.980000
dp                       -5.991469
qm1                       2.302564
kl1                       9.210334
sorbent_cost_per_kg       7.600902
total_LCP                29.519682
emissions                19.495519
PFAS_eff            

In [31]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

Number of rows after cleaning: 200000


In [32]:
# separate the data into inputs and outputs
inputs = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    ]

outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff"
]

dfin = df[inputs]
dfout = df[outputs]

In [33]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [34]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [35]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [36]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [37]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [38]:
# Optimisation Data
obj_scaler = 1e-6

#placeholder input parameters for the optimization model
# process parameters
feed_conc_val = 0.05 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 1000

# materials properties
density_val = 750 # kg/m3
porosity_val = 0.4
dp_val = 0.002 # m
qm1_val = 0.5 # mol/kg
kl1_val = 10 # m3/mol
sorbent_cost_val = 2 # $/kg

# loading saved nn
nn_PFAS_model = keras.models.load_model("ALLMAT_relu_mse.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, density=density_val, porosity=porosity_val, dp=dp_val, qm1=qm1_val, kl1=kl1_val, sorbent_cost=sorbent_cost_val):

    max_capacity_per_bed = 120 # m3/h
    N_beds = int(np.ceil(capacity/max_capacity_per_bed))
    Q_per_column = capacity/N_beds

    optimization_model = pyo.ConcreteModel()
    
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.dp = pyo.Var(initialize=(unscaled_lb[7]+unscaled_ub[7])/2, bounds=(unscaled_lb[7], unscaled_ub[7]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    optimization_model.nn = OmltBlock()
    formulated_problem = ReluBigMFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    optimization_model.nn.inputs[0].fix(pyo.log(capacity))
    optimization_model.nn.inputs[5].fix(pyo.log(feed_conc))
    optimization_model.nn.inputs[6].fix(density)
    optimization_model.nn.inputs[8].fix(pyo.log(qm1))
    optimization_model.nn.inputs[9].fix(pyo.log(kl1))
    optimization_model.nn.inputs[10].fix(pyo.log(sorbent_cost))

    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_dp = pyo.Constraint(expr= optimization_model.dp == optimization_model.nn.inputs[7])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    optimization_model.min_velocity = pyo.Constraint(expr= 4*Q_per_column >= 2*(np.pi*optimization_model.diameter**2))
    optimization_model.max_velocity = pyo.Constraint(expr= 4*Q_per_column <= 15*(np.pi*optimization_model.diameter**2))

    if eps_emissions is not None:
        if eps_emissions <= 0:
            raise ValueError(f"eps_emissions must be positive, got {eps_emissions}")
        # The network works in log space, so the budget must be logged too
        optimization_model.emissions_con = pyo.Constraint(
            expr= optimization_model.emissions <= np.log(eps_emissions))
    
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr=(optimization_model.total_LCP), sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr=(optimization_model.emissions), sense=pyo.minimize)

    options = {
    'MaxTime': 500,
    'epsr': 1e-4,  
    'epsa': 1e-3,
    'lpsol': 3,
    'cplexlibname': '"C:/Users/huberttreve/Documents/Python/Solvers/CPLEX/cplex/bin/x64_win64/cplex2212.dll"',
    'threads': 4,
    'prlevel': 1,
    'dolocal': 1
    }

    # print("# of variables: ", optimization_model.nvariables())
    # print("# of constraints: ", optimization_model.nconstraints())
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=False, options=options)
    
    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        # print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        # print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        # print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            N_beds,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            np.exp(get_val(optimization_model.dp)),
            np.exp(get_val(optimization_model.ads_time)),
            feed_conc,
            density,
            qm1,
            kl1,
            sorbent_cost,
            np.exp(get_val(optimization_model.total_LCP)),
            np.exp(get_val(optimization_model.emissions)),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [39]:
import time
from pathlib import Path

MW_PFAS = 414.07  # g/mol

def get_effluent_mol_m3(concentration_ppb):
    return (concentration_ppb * 1e-3) / MW_PFAS

# 1. Fixed operating conditions (same for every material)
fixed_capacity  = 1000     # m3/h
fixed_feed_conc = 0.5      # ppb, must lie inside the surrogate's training range
epa_limit_ppb   = 0.004    # 4 ppt PFOA limit
epa_mol_m3      = get_effluent_mol_m3(epa_limit_ppb)

# 2. Materials (file names are built from the keys, so keep them short and filename safe)
MATERIALS = {
    'GAC': dict(qm1=0.57, kl1=18, density=501, sorbent_cost=2),
    'Amberlite 400': dict(qm1=2.92, kl1=69, density=670, sorbent_cost=10),    # placeholder values
    'Amberlite™ IRA910': dict(qm1=3.47, kl1=416, density=700, sorbent_cost=12.5),   # placeholder values
}

# 3. Sweep settings
N_EPS = 20                  # Interior epsilon points between the two anchors
EPS_SPACING = 'log'      # 'linear' or 'log' (log suits fronts spanning orders of magnitude)
EPS_REL_RELAX = 1e-4        # Relative relaxation of the emissions minimum, avoids an infeasible endpoint
BINDING_REL_TOL = 1e-3      # Emissions within this fraction of epsilon count as a binding constraint
VALID_STATUSES = ['Optimal', 'Feasible (Time Limit)']
OUTPUT_FOLDER = Path("./pareto_data")
OVERWRITE = True           # False skips materials whose output file already exists (resume after a crash)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Column order must match the solve_model return tuple
columns = [
    "capacity", "N_beds", "diameter", "column_height", "porosity", "dp",
    "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
    "total_LCP", "emissions", "PFAS_eff_conc", "status"
]

def make_failed_res(params, status_msg):
    """Placeholder tuple: NaN outputs, inputs kept so failed runs stay traceable."""
    return (
        fixed_capacity, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        fixed_feed_conc, params['density'], params['qm1'], params['kl1'], params['sorbent_cost'],
        np.nan, np.nan, np.nan, status_msg
    )

def run(params, objective, eps=None):
    """One solve. Returns a dict of outputs plus solve time; never raises."""
    t0 = time.perf_counter()
    try:
        res = solve_model(
            objective=objective,
            eps_emissions=eps,
            capacity=fixed_capacity,
            feed_conc=fixed_feed_conc,
            EPA_conc=epa_mol_m3,
            qm1=params['qm1'],
            kl1=params['kl1'],
            density=params['density'],
            sorbent_cost=params['sorbent_cost']
        )
    except Exception as e:
        print(f"    ⚠️ Solver crashed ({e})")
        res = make_failed_res(params, "Solver Crashed")
    r = dict(zip(columns, res))
    r['solve_time_s'] = time.perf_counter() - t0
    r['eps_emissions'] = eps
    return r

def is_valid(r):
    return (r['status'] in VALID_STATUSES
            and np.isfinite(r['total_LCP']) and np.isfinite(r['emissions']))

def pareto_mask(cost, emis):
    """True for points not dominated by any other point (minimising both)."""
    cost, emis = np.asarray(cost), np.asarray(emis)
    keep = np.ones(len(cost), dtype=bool)
    for i in range(len(cost)):
        dominated = (cost <= cost[i]) & (emis <= emis[i]) & ((cost < cost[i]) | (emis < emis[i]))
        keep[i] = not dominated.any()
    return keep

# 4. Sweep each material
print(f"Pareto sweep: Q = {fixed_capacity} m3/h, C_o = {fixed_feed_conc} ppb, "
      f"EPA limit = {epa_limit_ppb} ppb, {len(MATERIALS)} materials")

for name, params in MATERIALS.items():
    out_file = OUTPUT_FOLDER / f"{name}.xlsx"
    if out_file.exists() and not OVERWRITE:
        print(f"\n[{name}] Output exists, skipping (set OVERWRITE = True to redo).")
        continue

    print(f"\n[{name}] qm1={params['qm1']}, kl1={params['kl1']}, "
          f"density={params['density']}, sorbent_cost={params['sorbent_cost']}")
    records = []

    # Anchor 1: minimum cost with no emissions constraint
    print("  Anchor: minimum cost")
    a_cost = run(params, objective='cost', eps=None)
    a_cost['run_type'] = 'anchor_min_cost'
    records.append(a_cost)

    # Anchor 2: minimum emissions
    print("  Anchor: minimum emissions")
    a_emis = run(params, objective='emissions', eps=None)
    a_emis['run_type'] = 'anchor_min_emissions'
    records.append(a_emis)

    if not (is_valid(a_cost) and is_valid(a_emis)):
        print("  Skipping sweep: an anchor run failed. Saving anchors only for inspection.")
        pd.DataFrame(records).to_excel(out_file, sheet_name='All_Runs', index=False)
        continue

    E_min, E_max = a_emis['emissions'], a_cost['emissions']
    print(f"  Emissions range: {E_min:.6g} to {E_max:.6g} kgCO2/h | "
          f"cost range: {a_cost['total_LCP']:.6g} to {a_emis['total_LCP']:.6g}")

    if E_max <= E_min * (1 + EPS_REL_RELAX):
        print("  Cost and emissions optima coincide: no trade-off, the front is a single point.")
        eps_values = []
    else:
        # Lexicographic low end: cheapest design at (just above) minimum emissions
        eps_low = E_min * (1 + EPS_REL_RELAX)
        if EPS_SPACING == 'log' and E_min > 0:
            interior = np.geomspace(eps_low, E_max, N_EPS + 2)[1:-1]
        else:
            interior = np.linspace(eps_low, E_max, N_EPS + 2)[1:-1]
        eps_values = np.concatenate([[eps_low], interior])

    # Epsilon-constraint sweep: minimise cost subject to emissions <= eps
    for j, eps in enumerate(eps_values, start=1):
        print(f"  [{j}/{len(eps_values)}] eps = {eps:.6g} kgCO2/h")
        r = run(params, objective='cost', eps=eps)
        r['run_type'] = 'eps_low_end' if j == 1 else 'eps'
        records.append(r)

    # Post-processing
    df = pd.DataFrame(records)
    df.insert(0, 'material', name)
    df['valid'] = df.apply(is_valid, axis=1)

    has_eps = df['eps_emissions'].notna()
    df['eps_binding'] = np.where(
        has_eps & df['valid'],
        df['emissions'] >= df['eps_emissions'] * (1 - BINDING_REL_TOL),
        np.nan
    )

    df['on_front'] = False
    v = df['valid']
    if v.any():
        df.loc[v, 'on_front'] = pareto_mask(df.loc[v, 'total_LCP'], df.loc[v, 'emissions'])

    front = (df[df['on_front']]
             .drop_duplicates(subset=['total_LCP', 'emissions'])
             .sort_values('emissions'))

    settings = pd.DataFrame([{
        'material': name, **params,
        'capacity_m3h': fixed_capacity, 'feed_conc_ppb': fixed_feed_conc,
        'epa_limit_ppb': epa_limit_ppb, 'epa_limit_mol_m3': epa_mol_m3,
        'E_min': E_min, 'E_max': E_max,
        'cost_at_E_min': a_emis['total_LCP'], 'cost_at_E_max': a_cost['total_LCP'],
        'n_eps': len(eps_values), 'eps_spacing': EPS_SPACING,
    }])

    with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
        front.to_excel(writer, sheet_name='Pareto_Front', index=False)
        df.to_excel(writer, sheet_name='All_Runs', index=False)
        settings.to_excel(writer, sheet_name='Settings', index=False)

    n_valid = int(df['valid'].sum())
    n_nonbinding = int((df['eps_binding'] == False).sum())
    print(f"  Done: {n_valid}/{len(df)} valid, {len(front)} on the front, "
          f"{n_nonbinding} non-binding eps runs. Saved {out_file}")

print("\n✅ Pareto sweep complete.")

Pareto sweep: Q = 1000 m3/h, C_o = 0.5 ppb, EPA limit = 0.004 ppb, 3 materials

[GAC] qm1=0.57, kl1=18, density=501, sorbent_cost=2
  Anchor: minimum cost
  Anchor: minimum emissions
  Emissions range: 888.357 to 956.023 kgCO2/h | cost range: 247813 to 276907
  [1/21] eps = 888.446 kgCO2/h
  [2/21] eps = 891.552 kgCO2/h
  [3/21] eps = 894.67 kgCO2/h
  [4/21] eps = 897.799 kgCO2/h
  [5/21] eps = 900.938 kgCO2/h
  [6/21] eps = 904.089 kgCO2/h
  [7/21] eps = 907.251 kgCO2/h
  [8/21] eps = 910.423 kgCO2/h
  [9/21] eps = 913.607 kgCO2/h
  [10/21] eps = 916.802 kgCO2/h
  [11/21] eps = 920.008 kgCO2/h
  [12/21] eps = 923.225 kgCO2/h
  [13/21] eps = 926.453 kgCO2/h
  [14/21] eps = 929.693 kgCO2/h
  [15/21] eps = 932.944 kgCO2/h
  [16/21] eps = 936.207 kgCO2/h
  [17/21] eps = 939.481 kgCO2/h
  [18/21] eps = 942.766 kgCO2/h
  [19/21] eps = 946.063 kgCO2/h
  [20/21] eps = 949.371 kgCO2/h
  [21/21] eps = 952.691 kgCO2/h
  Done: 23/23 valid, 23 on the front, 0 non-binding eps runs. Saved pareto_dat